### Structure


In [7]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:openai/gpt-oss-20b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022BECE72D50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022BECE71A90>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

### Pyantic

In [8]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="title of the movie")
    year:int=Field(description="year the move was released")
    director:str=Field(description="director of the movie")
    rating:float=Field(description="movie rating out of 10")

model_with_structure=model.with_structured_output(Movie)

response=model_with_structure.invoke("Tell me about movie avengers endgame")

response

Movie(title='Avengers: Endgame', year=2019, director='Anthony Russo', rating=8.4)

In [9]:
#pydantic can also include simple raw message along with structure

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response=model_with_structure.invoke("Tell me about movie avengers endgame")

response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to output a response. The user wants information about the movie "Avengers: Endgame". We can use the function. The function requires director, rating, title, year. We\'ll call the function.', 'tool_calls': [{'id': 'fc_cfe82666-1ef6-428d-8344-dec2cf0574e5', 'function': {'arguments': '{"director":"Anthony Russo, Joe Russo","rating":8.4,"title":"Avengers: Endgame","year":2019}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 155, 'total_tokens': 244, 'completion_time': 0.112423193, 'completion_tokens_details': {'reasoning_tokens': 44}, 'prompt_time': 0.007470675, 'prompt_tokens_details': None, 'queue_time': 0.314837221, 'total_time': 0.119893868}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17

In [10]:
#nested pydantic structure

class actor(BaseModel):
    name:str
    role:str

class movie(BaseModel):
    title:str
    year:int
    cast:list[actor]
    genres:list[str]
    budget:float | None=Field(None,description="Budget In Millions USD")

model_with_structure=model.with_structured_output(movie)

response=model_with_structure.invoke("Tell me about movie avengers endgame")

response

movie(title='Avengers: Endgame', year=2019, cast=[actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), actor(name='Chris Evans', role='Steve Rogers / Captain America'), actor(name='Mark Ruffalo', role='Bruce Banner / Hulk'), actor(name='Chris Hemsworth', role='Thor'), actor(name='Scarlett Johansson', role='Natasha Romanoff / Black Widow'), actor(name='Jeremy Renner', role='Clint Barton / Hawkeye'), actor(name='Don Cheadle', role='James Rhodes / War Machine'), actor(name='Paul Rudd', role='Scott Lang / Ant-Man'), actor(name='Brie Larson', role='Carol Danvers / Captain Marvel'), actor(name='Benedict Cumberbatch', role='Stephen Strange / Doctor Strange'), actor(name='Josh Brolin', role='Thanos'), actor(name='Elizabeth Olsen', role='Hawkeye')], genres=['Action', 'Adventure', 'Science Fiction'], budget=356000000.0)

### TypedDict

In [ ]:
from typing_extensions import TypedDict,Annotated

class movie(TypedDict):
    """a movie with details"""

    title: Annotated[str,...,"title of the movie"] #... is optional field
    year: Annotated[int,...,"year the move was released"]
    director: Annotated[str,...,"director of the movie"]
    rating: Annotated[float,...,"movie rating out of 10"]

model_with_structure=model.with_structured_output(movie)

response=model_with_structure.invoke("Tell me about avengers endgame")

response

{'director': 'Anthony Russo and Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [14]:
#nested TypedDict structure
class actor(TypedDict):
    name:str
    role:str

class movie(TypedDict):
    title:str
    year:int
    cast:list[actor]
    genres:list[str]
    budget:float | None=Field(None,description="Budget In Millions USD")

model_with_structure=model.with_structured_output(movie)

response=model_with_structure.invoke("Tell me about movie avengers endgame")

response

{'budget': 356000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Don Cheadle', 'role': 'James Rhodes / War Machine'},
  {'name': 'Paul Rudd', 'role': 'Scott Lang / Ant-Man'},
  {'name': 'Brie Larson', 'role': 'Carol Danvers / Captain Marvel'},
  {'name': 'Tom Holland', 'role': 'Peter Parker / Spider-Man'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers: Endgame',
 'year': 2019}

### DataClasses

In [16]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str=Field(description="The name of the person")
    email: str=Field(description="The email address of the person")
    phone: str=Field(description="The phone number of the person")


model=init_chat_model("groq:openai/gpt-oss-20b")

agent=create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result=agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='d5cb12c1-6930-4191-ad4e-28d79ab5ffda'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'We need to output JSON object with fields name, email, phone. Must be valid compact JSON. Use the schema. Ensure all required fields present. Output only the JSON object.'}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 238, 'total_tokens': 311, 'completion_time': 0.075934, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.016315127, 'prompt_tokens_details': None, 'queue_time': 0.316104208, 'total_time': 0.092249127}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1748

In [17]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')